# Detection Thresholds and CV0-Based Responses

Stage 03 is the NEURON/MRG stage of the manuscript pipeline. The release notebooks do not rerun NEURON; instead, this notebook explains the logic and visualizes the archived stage-03 outputs.


## Pipeline Map

For each stage-02 `V(t, node)` repetition, stage 03:

1. Plays the extracellular potential into the MRG axon model in NEURON.
2. Searches for spike-count threshold amplitudes (`I_first`, volley thresholds, percent-of-pulse thresholds, and `I_max`).
3. Builds fixed CV0-based amplitudes for downstream comparisons.
4. Replays every repetition at those CV0-based amplitudes and records spike counts and spike times.

The cached CSV outputs are the handoff into feature engineering and behavioral discrimination.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
import matplotlib.pyplot as plt

def _bootstrap_src(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        src_dir = candidate / 'src'
        if src_dir.exists() and (candidate / 'data').exists():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = _bootstrap_src(Path.cwd())

from utils import require_data_files, data_subdir, ensure_src_on_path, find_repo_root
from figures import set_manuscript_style

REPO_ROOT = find_repo_root(REPO_ROOT)
SRC_DIR = ensure_src_on_path(REPO_ROOT)
require_data_files(
    REPO_ROOT,
    [
        'data/time-patterns/generated',
        'data/space-time-V/generated',
    ],
    context='03_DetectionThresholds',
)


from detection import (
    DEFAULT_CV_VALUES,
    DEFAULT_FREQUENCIES_HZ,
    DetectionConfig,
    load_detection_tables,
    load_detection_response_tables,
    load_input_pulse_onsets_s,
    make_detection_response_comparison_widget,
    parse_numeric_list,
    plot_cv0_response_counts,
    plot_spike_count_threshold_amplitudes,
    plot_threshold_response_raster,
    plot_threshold_search_trace,
    plot_threshold_spike_counts,
    select_detection_row,
    threshold_summary_for_row,
    verify_detection_outputs,
)

set_manuscript_style(dpi=120)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)

## Configure and Check Data

The data check below runs silently unless an expected stage-03 file is missing.


In [ ]:
space_time_dir = data_subdir(REPO_ROOT, 'space_time') / 'generated'
time_patterns_dir = data_subdir(REPO_ROOT, 'time_patterns') / 'generated'
detection_dir = data_subdir(REPO_ROOT, 'detection') / 'generated'
config = DetectionConfig(
    cv_vals=DEFAULT_CV_VALUES,
    freq_vals=DEFAULT_FREQUENCIES_HZ,
    cutline=101,
    spacetime_dir=space_time_dir,
    time_pattern_dir=time_patterns_dir,
    output_dir=detection_dir,
    detection_csv=detection_dir / 'detection_thresholds.csv',
    cv0_threshold_csv=detection_dir / 'cv0_threshold_max.csv',
    detection_responses_csv=detection_dir / 'detection_responses.csv',
    cv0_responses_csv=detection_dir / 'cv0_based_responses.csv',
).resolved()

status = verify_detection_outputs(config)
if not status.outputs_complete:
    missing = list(status.missing_output_files)
    preview = chr(10).join(str(path) for path in missing[:8])
    raise FileNotFoundError('Stage-03 cached outputs are missing. First missing files:' + chr(10) + preview)

df_thresholds, df_cv0_thresholds, df_cv0_responses = load_detection_tables(config)
df_detection_responses, df_cv0_responses = load_detection_response_tables(config)

## Step 1: Rep-Matched Threshold Responses

Use the dropdowns to compare three cached response views for the same frequency and repetition: the regular CV0 response at its true thresholds, the selected irregular CV response at its own true thresholds, and the selected irregular CV response replayed at the rep-matched CV0 amplitudes. This shows the transition from threshold finding to the CV0-based response data used by downstream feature engineering.


In [ ]:
response_widget = make_detection_response_comparison_widget(
    df_detection_responses=df_detection_responses,
    df_cv0_responses=df_cv0_responses,
    config=config,
    default_freq=100.0,
    default_cv=0.7,
    default_rep_index=5,
)
display(response_widget)

## Step 2: Manuscript-Style Threshold Summaries

These panels summarize the full archived stage-03 output across CV and frequency. The shaded regions are SEM across repetitions. Absolute spike-count thresholds are plotted as counts; percent-of-pulse thresholds are plotted as percentages of expected pulses.


In [ ]:
fig, axes = plot_spike_count_threshold_amplitudes(df_thresholds, cutline=config.cutline)
display(fig)
plt.close(fig)

fig, axes = plot_threshold_spike_counts(df_thresholds, cutline=config.cutline)
display(fig)
plt.close(fig)

fig, axes = plot_cv0_response_counts(df_cv0_responses, cutline=config.cutline)
display(fig)
plt.close(fig)

---